In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os

def get_data(filename='L64.png'):
    """image dataを得る。

    Returns:
        [np.array]: image data.
        str: path to the image data.
    """
    # https://commons.wikimedia.org/wiki/File:Old_English_typeface.svg
    # から文字'L'は取得しました。

    prefix = "../data/font"
    file = os.path.join(prefix,filename)

    # 画像の読み込み
    im = Image.open(file)

    #im = im.resize((128,128))

    # 画像をarrayに変換
    im_list = np.array(im)
    print(im_list.shape)  # (R,G,B,alpha)と４要素入っている。
    return im_list, prefix


TARGET_FILENAME = 'L64.png'
g_im_list, g_prefix = get_data(TARGET_FILENAME)

# 表示
plt.imshow(g_im_list)


画像データの確認。

白黒だがRGBデータである。

In [ ]:
g_im_list.shape


In [ ]:
def make_image(im_list, m1=0):
    """    濃淡変換を行う。
    m1で調整する。m1=0だと調整しない。

    論文で見栄えを良くするため、結果をよく見せるため、
    この変換を行うとデータの捏造になるらしいので注意。

    Args:
        im_list (np.array): image (R,G,B,alpha)
        m1 (int, optional): 255.0/(value-m1)で濃淡変換。 Defaults to 0.

    Returns:
        np.array: 変換されたimage
    """

    # 反転
    im_list = im_list.max() - im_list
    print("image BW", im_list.min(), im_list.max())

    center = np.array(im_list.shape)
    for i in range(center.shape[0]):
        center[i] /= 2.0
    center = center[:2]
    print("center", center)
    valuem = im_list.max()

    m1v = np.array([m1, m1, m1])

    r2 = (center[0]*0.9)**2

    new_im = []
    for i, x in enumerate(im_list):
        rlist = []
        for j, r in enumerate(x):
            v = [i, j] - center
            r = r[:3]
            r = r.astype(np.float64) - m1v

            r *= 255.0/(valuem-m1)
            if r[0] < 0:
                # print(r)
                r0 = 0.0
                r = [r0, r0, r0]
                # print(r)
            elif r[0] > 255:
                # print(r)
                r0 = 255
                r = [r0, r0, r0]
            if np.sum(v*v) > r2:
                r = [0, 0, 0]
            rlist.append(r)
        new_im.append(rlist)
    new_im = np.array(new_im).astype(np.uint8)
    print(new_im.min(), new_im.max())
    return new_im


g_new_im = make_image(g_im_list)


表示

In [ ]:
plt.imshow(g_new_im)


In [ ]:
# alpha channelを除いた。
g_new_im.shape


In [ ]:
def make_mat(new_im2):
    """RGBの3成分をgrayscaleとして１成分に変換する。

    Args:
        new_im2 (np.array): RBG image

    Returns:
        np.array: 変換された白黒image
    """

    mat = []
    for x in new_im2:
        rlist = []
        for y in x:
            rlist.append(y[0])
        mat.append(rlist)
    mat = np.array(mat)
    print(mat.min(), mat.max())
    return mat


def plot_hist(mat, bins=20):
    """濃淡のhistgram表示

    Args:
        mat (np.array): image
        bins (int, optional): # of bins. Defaults to 20.
    """
    print(mat.min(), mat.max())
    valuem = mat.max()

    plt.figure()
    plt.subplot(121)
    plt.hist(mat.ravel(), bins=bins)
    # 0-maxまでを表示
    plt.xlim((0, valuem))
    plt.subplot(122)
    plt.hist(mat.ravel(), bins=bins)
    # 50-maxまでを表示
    plt.xlim((50, valuem))
    plt.show()


In [ ]:
g_mat = make_mat(g_new_im)
plot_hist(g_mat)


In [ ]:
# save file
def save_csv(prefix, target_filename):
    target_filename = os.path.splitext(target_filename)[0]+".csv"
    filepath = os.path.join(prefix, target_filename)
    if not os.path.isfile(filepath):
        np.savetxt(filepath+"L64.csv", g_mat, delimiter=",", fmt="%d")
        print(filepath,"is mde.")
        
save_csv(g_prefix, TARGET_FILENAME)